In [53]:
import os
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import asyncio
from mcp import ClientSession
from mcp.client.sse import sse_client
import os
from dotenv import load_dotenv
from groq import Groq
import json

load_dotenv(override=True)

True

In [3]:
MCP_URL = os.environ.get("MCP_SERVER_URL", "http://127.0.0.1:8150/sse")
MCP_URL

'http://127.0.0.1:8150/sse'

In [4]:
client = Groq()

In [65]:
def convert_to_schema(tool) -> dict:
    """
    Convert MCP-style tool schema to OpenAI function calling format.
    """

    return {
        "type": "function",
        "function": {
            "name": tool.name,
            "description": tool.description.strip(),
            "parameters": {
                "type": "object",
                "properties": tool.inputSchema["properties"],
                "required": tool.inputSchema.get("required", []),
            },
        },
    }

In [77]:
async with sse_client(MCP_URL) as (read, write):
    async with ClientSession(read, write) as session:
        # initialize connection
        await session.initialize()
        
        # 1. list tools
        tools = await session.list_tools()
        schemas = []
        for tool in tools.tools:
            schemas.append(convert_to_schema(tool))

        messages = [{"role": "user", "content": "How many tables in the database?"}]
        response = client.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=messages,
            tools=schemas
        )   
        messages.append(response.choices[0].message)

        async def execute_tool_call(tool_call):
            """Parse and execute a single tool call"""
            function_name = tool_call.function.name
            # function_to_call = available_functions[function_name]
            function_args = json.loads(tool_call.function.arguments)
        
            print(function_name)
            print(function_args)
        
            return await session.call_tool(
                function_name,
                **function_args
            ) 
            
        if response.choices[0].message.tool_calls:
            for tool_call in response.choices[0].message.tool_calls:
                function_response = await execute_tool_call(tool_call)
                
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": tool_call.function.name,
                    "content": str(function_response)
                })   

list_tables
{}


In [78]:
final = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=messages
)

In [79]:
final.choices[0].message.content

'The database currently contains **1 table**.'